<a href="https://colab.research.google.com/github/kfartem434-png/MN_Kapusta/blob/main/%D0%9B%D0%A0_9_%D0%9A%D0%B0%D0%BF%D1%83%D1%81%D1%82%D0%B0_3_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** __________________________  
**Група:** _________________________________  
**Дата:** __________________________________

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [78]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [79]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.16
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [80]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [81]:
# TODO 2.1 — Дослідіть набір даних:
# - кількість унікальних користувачів;
# - кількість унікальних фільмів;
# - мінімальне, максимальне, середнє значення rating;
# - пропущені значення (за наявності).

# Ваш код:
print("Кількість унікальних користувачів:",len(ratings_df["user_id"].value_counts()))
print("Кількість унікальних фільмів:",len(ratings_df["item_id"].value_counts()))
print(
  "Rating:\n",
  "мінімальне:",ratings_df["rating"].min(),
  "максимальне:",ratings_df["rating"].max(),
  "середнє:", ratings_df["rating"].mean()
)
print("Пропуски:")
display(ratings_df.isna().sum())

Кількість унікальних користувачів: 943
Кількість унікальних фільмів: 1682
Rating:
 мінімальне: 1.0 максимальне: 5.0 середнє: 3.52986
Пропуски:


,0
user_id,0
item_id,0
rating,0
timestamp,0


**Короткі спостереження щодо даних:**

_Запишіть 2–3 речення про кількість користувачів, фільмів та характер оцінок._

Кількість унікальних користувачів у датасеті приблизно у два рази меньше ніж кількість унікальних фільмів, тобто на кожного користувача припадає у середньому дві рецензії фільмів. Оцінки фільмів знаходяться у заданому діапазоні від 1 до 5, середнє значення оцінок є вищим за медіану діапазону на одну одиницю.

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [82]:
# TODO 3.1 — Створіть train_df та test_df із ratings_df.
# Використайте sklearn_train_test_split(..., test_size=0.2, random_state=42).

# Ваш код:

train_df, test_df = sklearn_train_test_split(ratings_df,test_size=0.2,random_state=42)

In [83]:
# TODO 3.2 — Підготуйте об’єкти для Surprise:
# 1. reader = Reader(rating_scale=...)
#    Підказка: data.reader.rating_scale
# 2. train_data = Dataset.load_from_df(...)
#    Візьміть тільки user_id, item_id, rating із train_df.
# 3. testset = список кортежів (user_id, item_id, rating) із test_df.
# 4. Перевірте розміри train/test.

# Ваш код:

reader = Reader(rating_scale=(1, 5))
train_data = Dataset.load_from_df(train_df[["user_id","item_id","rating"]],reader)
testset = list(map(tuple,test_df[["user_id","item_id","rating"]].values))

print("train len:",len(train_data.df))
print("test len:",len(testset))

train len: 80000
test len: 20000


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [84]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [20,50, 100],
    'lr_all': [0.005, 0.010,0.001],
    'reg_all': [0.02, 0.05,0.1]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

_Поясніть, які параметри впливають на якість моделей і чому обрали саме ці значення._

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [85]:
# TODO 5.1 — Підберіть гіперпараметри SVD.
# Підказка: GridSearchCV(SVD, param_grid_svd,
#                        measures=['rmse', 'mae'], cv=3, n_jobs=-1)
# Після створення викличте .fit(train_data).

# Ваш код:

grid_search_svd = GridSearchCV(SVD, param_grid_svd, measures=["rmse", "mae"], cv=3, n_jobs=-1)
grid_search_svd.fit(train_data)

In [86]:
# TODO 5.2 — Підберіть гіперпараметри KNNBasic.
# Підказка: використайте param_grid_knn та ті самі метрики/cv.

# Ваш код:

grid_search_knn = GridSearchCV(KNNBasic,param_grid_knn,measures=["rmse","mae"],cv=3,n_jobs=-1)
grid_search_knn.fit(train_data)

In [92]:
# TODO 5.3 — Виведіть найкращий RMSE, MAE та параметри обох моделей.
# Підказки:
# grid_search_svd.best_score['rmse']
# grid_search_svd.best_params['rmse']
# grid_search_knn.best_score['rmse']

# Ваш код:
print("Best score SVD:\n MAE:",grid_search_svd.best_score["mae"],"RMSE:",grid_search_svd.best_score["rmse"])
print("Best parameters SVD:\n ",grid_search_svd.best_params["rmse"])
print()

print("Best score KNN:\n MAE:",grid_search_knn.best_score["mae"],"RMSE:",grid_search_knn.best_score["rmse"])
print("Best parameters KNN:\n ",grid_search_knn.best_params["rmse"])

Best score SVD:
 MAE: 0.7409144147132442 RMSE: 0.9366282054271201
Best parameters SVD:
  {'n_factors': 100, 'lr_all': 0.01, 'reg_all': 0.1}

Best score KNN:
 MAE: 0.8155209617758131 RMSE: 1.0283009431688943
Best parameters KNN:
  {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
|:--|--:|--:|:--|
| SVD | ... | ... | ... |
| KNNBasic | ... | ... | ... |

_Порівняйте якість моделей на перехресній перевірці. Зазначте, які параметри стали найкращими._

В цілому, кращий результат показала модель SVD, яка має на пів одиниці менші метрики MAE та RMSE ніж KNNBasic. Результатом такої оцінки є підбір параметрів n_factors = 100, lr_all = 0.01 та reg_all = 0.1

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [88]:
# TODO 6.1 — Отримайте best_svd_params і best_knn_params за CV RMSE.
# TODO 6.2 — Створіть trainset із train_data.
# TODO 6.3 — Навчіть best_svd і best_knn.
# Для відтворюваності задайте random_state=42 для SVD.

# Ваш код:

trainset = train_data.build_full_trainset()

best_svd = SVD(**grid_search_svd.best_params["rmse"],random_state=42)
best_svd.fit(trainset)

best_knn = KNNBasic(**grid_search_knn.best_params["rmse"])
best_knn.fit(trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [89]:
# TODO 7.1 — Отримайте svd_predictions та knn_predictions.
# TODO 7.2 — Обчисліть svd_rmse, svd_mae, knn_rmse, knn_mae.
# Підказка: accuracy.rmse(predictions, verbose=False)
#           accuracy.mae(predictions, verbose=False)

# Ваш код:

svd_predictions = best_svd.test(testset)

knn_predictions = best_knn.test(testset)

metrics = dict()

metrics["svd"] = {"RMSE": accuracy.rmse(svd_predictions),"MAE": accuracy.mae(svd_predictions)}

metrics["knn"] = {"RMSE": accuracy.rmse(knn_predictions),"MAE": accuracy.mae(knn_predictions)}

RMSE: 0.9196
MAE:  0.7275
RMSE: 1.0120
MAE:  0.7984


In [90]:
# TODO 7.3 — Сформуйте DataFrame results із колонками:
# Model, RMSE, MAE.
# Відсортуйте за RMSE та покажіть порівняння.

# Ваш код:
results = pd.DataFrame(metrics).T
results.index.name = "Model"
display(results)

,RMSE,MAE
Model,,
svd,0.919643,0.727531
knn,1.011953,0.798421


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
|:--|--:|--:|
| SVD | ... | ... |
| KNNBasic | ... | ... |

**Порівняння з CV:** _Прокоментуйте, чи суттєво відрізняються оцінки на кросвалідації та незалежній тестовій вибірці._

При порівняні результату оцінки на незалежній тестовії вибірці з результатом кросвалідації, можна побачити що результат не сильно змінився.

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [91]:
# Початковий ID можна змінити
user_id = "196"

# TODO 8.1 — Визначте найкращу за CV RMSE модель.
# TODO 8.2 — Перевірте, що користувач присутній у trainset.
# TODO 8.3 — Знайдіть неоцінені ним фільми.
# TODO 8.4 — Спрогнозуйте оцінки та покажіть топ-10.
# Бажані колонки результату: item_id, predicted_rating.
# Передбачте ситуацію, коли обраного user_id немає у trainset.

# Ваш код:

print("найкраща модель за CV RMSE -- це SVD")
best_model = best_svd
best_model.fit(trainset)

if trainset.knows_user(int(user_id)):
  print(f"Користувач {user_id} присутній у trainset")

  inner_uid = trainset.to_inner_uid(user_id)
  user_ratings = trainset.ur[inner_uid]

  all_items = set(trainset.all_items())
  rated_items = set([item for (item, _) in user_ratings])
  unrated_items = all_items - rated_items

  predictions = [
    (item, best_model.predict(user_id, trainset.to_raw_iid(item)).est)
    for item in unrated_items
  ]
  predictions.sort(key=lambda x: x[1], reverse=True)

  print()
  print("Топ-10 фільмів, рекомендованих для користувача:")
  for item_id, rating in predictions[:10]:
    print(f"Фільм {trainset.to_raw_iid(item_id)} з прогнозованим рейтингом {rating:.2f}")
else:
  print(f"Користувач {user_id} не присутній у trainset")

найкраща модель за CV RMSE -- це SVD
Користувач 196 присутній у trainset

Топ-10 фільмів, рекомендованих для користувача:
Фільм 318 з прогнозованим рейтингом 4.41
Фільм 64 з прогнозованим рейтингом 4.39
Фільм 1169 з прогнозованим рейтингом 4.35
Фільм 1449 з прогнозованим рейтингом 4.33
Фільм 114 з прогнозованим рейтингом 4.31
Фільм 320 з прогнозованим рейтингом 4.31
Фільм 603 з прогнозованим рейтингом 4.31
Фільм 427 з прогнозованим рейтингом 4.30
Фільм 483 з прогнозованим рейтингом 4.28
Фільм 513 з прогнозованим рейтингом 4.27


## 9. Перевірка виконання

Позначте після реалізації:

- [*] Дані `ml-100k` завантажено та перевірено.
- [*] Є розбиття 80/20 з фіксованим `random_state`.
- [*] `Reader`, `train_data` і `testset` створені коректно.
- [*] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [*] Для SVD і KNNBasic показано параметри та результати CV.
- [*] Моделі навчено на повній навчальній частині.
- [*] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [*] Наведено порівняльну таблицю та інтерпретацію.
- [*] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [*] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?
2. Які найкращі гіперпараметри отримано для обох моделей?
3. Які підсумкові значення RMSE та MAE на тестовій вибірці?
4. Чи є різниця між результатами CV та тестування?
5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

**Висновок:** _Напишіть 5–8 речень з опорою на отримані числові результати._

---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.

За кросвалідацією обрано модель SVD, так як вона показує найкращі показники у обох метриках RSME та MAE. Для обох моделей були підібрані гіперпараметри за допомогою функції GridSearchCV, з наступними результатами: для SVD було підібрано кількість факторів 100, вісоток навчання всіх параметрів 1% та строк регуляризації 10%; для KNN було підібрано наступні гіперпараметри: кількість сусідів 40, мінімальна кількість сусідів 1 та налаштування подібності, яке використовує косинусевий коефіцієнт кореляції на основі користувачаів.
Після повторної перевірки метрик на тестових даних метрики майже не змінилися, але зменшилися для обох моделей.

**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**